## Disappearing Cargo

- The warehouse manager is panicking. Show me exactly what's gone. List the `po_id` and `item_description` for every purchase order that has no corresponding delivery record, sorted by `po_id`.

In [ ]:
SELECT po.po_id, po.item_description
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
WHERE d.delivery_id IS NULL
ORDER BY po.po_id

- We suspect ghost vendors. Expose them by listing the `vendor_name` and `item_description` for every order that was never delivered, sorted by `vendor_name`.

In [ ]:
SELECT po.vendor_name, po.item_description
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
WHERE d.delivery_id IS NULL
ORDER BY po.vendor_name

- The Chief needs a number. Quantify the loss by calculating the total value of all undelivered orders (quantity * unit_price) and alias it as `missing_value`.

In [ ]:
SELECT SUM(po.quantity * po.unit_price) AS missing_value
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
WHERE d.delivery_id IS NULL

- Interrogation revealed they are skimming off the top. Detect this by listing the `po_id`, `item_description`, `quantity`, and `delivered_quantity` for any order where the delivered amount is less than ordered. Sort by `po_id`.

In [ ]:
SELECT po.po_id, 
    po.item_description, 
    po.quantity, 
    d.delivered_quantity
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
WHERE d.delivered_quantity < po.quantity
ORDER BY po.po_id

- It's an inside job. Trace the internal theft by listing the `delivery_id` and `item_description` for any shipment that was delivered but never scanned into inventory. Sort by `delivery_id`.

In [ ]:
SELECT d.delivery_id,
    po.item_description
FROM purchase_orders AS po
JOIN deliveries AS d
    ON po.po_id = d.po_id
LEFT JOIN inventory AS i
    ON d.po_id = i.po_id
WHERE i.po_id IS NULL
ORDER BY d.delivery_id

- The books are cooked. Audit the stock levels by listing the `item_description`, `expected_stock`, and `current_stock` for items where `current_stock` is less than expected. Calculate the difference as `missing`, sorted by `item_description`.

In [ ]:
SELECT po.item_description,
    i.expected_stock,
    i.current_stock,
    (i.expected_stock - i.current_stock) AS missing
FROM purchase_orders AS po
JOIN inventory AS i
    ON po.po_id = i.po_id
WHERE i.current_stock < i.expected_stock
ORDER BY po.item_description

- The books are cooked. Audit the stock levels by listing the `item_description`, `expected_stock`, and `current_stock` for items where `current_stock` is less than expected. Calculate the difference as `missing`, sorted by `item_description`.

In [ ]:
SELECT po.item_description, 
    i.expected_stock, i.current_stock, 
    (i.expected_stock - i.current_stock) AS missing 
FROM purchase_orders AS po 
JOIN inventory AS i 
    ON po.po_id = i.po_id 
WHERE i.current_stock < i.expected_stock 
ORDER BY po.item_description

- We need to find when the breakdown started. Analyze the timeline gaps by listing the `vendor_name`, `order_date`, and `delivery_date` for every purchase order, sorted by `order_date`.

In [ ]:
SELECT po.vendor_name,
    po.order_date,
    d.delivery_date
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
ORDER BY po.order_date

- Pinpoint the leaks. Rank the warehouses by loss, showing the `target_warehouse` and the count of **missing** deliveries (`missing_shipments`) for each. Order by `missing_shipments` from highest to lowest.

In [ ]:
SELECT po.target_warehouse,
    COUNT(*) AS missing_shipments 
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
WHERE d.delivery_id IS NULL
GROUP BY target_warehouse
ORDER BY missing_shipments DESC

- We need a performance review. Scorecard the vendors by listing the `vendor_name`, `total_orders`, `successful_deliveries`, and `total_value`. Rank by successful deliveries, highest to lowest.

In [ ]:
SELECT po.vendor_name, 
    COUNT(po.po_id) AS total_orders,
    COUNT(d.delivery_id) AS successful_deliveries,
    SUM(po.quantity * po.unit_price) AS total_value
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
GROUP BY po.vendor_name
ORDER BY successful_deliveries DESC

- The DA needs everything. Compile the final audit report listing `po_id`, `vendor_name`, `item_description`, `order_value`, `delivery_status`, and `inventory_loss`. Rank by order value descending.

In [ ]:
SELECT po.po_id, 
    po.vendor_name,
    po.item_description,
    (po.quantity * po.unit_price) AS order_value,
    (CASE 
        WHEN d.delivery_id IS NULL THEN 'Not Delivered'
        ELSE 'Delivered' 
    END) AS delivery_status,
    COALESCE(i.expected_stock - i.current_stock, 0) AS inventory_loss
FROM purchase_orders AS po
LEFT JOIN deliveries AS d
    ON po.po_id = d.po_id
LEFT JOIN inventory AS i
    ON d.po_id = i.po_id
ORDER BY order_value DESC